In [218]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import r2_score 
from sklearn.linear_model import SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import category_encoders as ce
from scipy.stats import ttest_rel 


Считываем файлы

In [220]:
folder = Path('cars')
files = {
    'audi': 'cars/audi.csv',
    'bmw': 'cars/bmw.csv',
    'cclass': 'cars/cclass.csv',
    'focus': 'cars/focus.csv',
    'ford': 'cars/ford.csv',
    'hyundi': 'cars/hyundi.csv',    
    'merc': 'cars/merc.csv',
    'skoda': 'cars/skoda.csv',    
    'toyota': 'cars/toyota.csv',    
    'vauxhall': 'cars/vauxhall.csv',    
    'vw': 'cars/vw.csv'    
}
# Список для хранения датафреймов
dfs = []
for brand, path in files.items():
    df = pd.read_csv(path)
    df['brand'] = brand
    dfs.append(df)
    
# Объединяем все датафреймы
resultDF = pd.concat(dfs, ignore_index=True)

In [221]:
#'tax(£)' столбец содержит значения только для hyundi, значит он не информативен для модели
resultDF = resultDF.drop(columns=['tax(£)'])

In [222]:
#пока без этих классов
df1 = pd.read_csv('cars/unclean cclass.csv')
df2 = pd.read_csv('cars/unclean focus.csv')

In [223]:
X = resultDF.drop(columns='price')
y = resultDF['price']

In [226]:
X.head(10)

,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,brand
0,A1,2017,Manual,15735,Petrol,150.0,55.4,1.4,audi
1,A6,2016,Automatic,36203,Diesel,20.0,64.2,2.0,audi
2,A1,2016,Manual,29946,Petrol,30.0,55.4,1.4,audi
3,A4,2017,Automatic,25952,Diesel,145.0,67.3,2.0,audi
4,A3,2019,Manual,1998,Petrol,145.0,49.6,1.0,audi
5,A1,2016,Automatic,32260,Petrol,30.0,58.9,1.4,audi
6,A6,2016,Automatic,76788,Diesel,30.0,61.4,2.0,audi
7,A4,2016,Manual,75185,Diesel,20.0,70.6,2.0,audi
8,A3,2015,Manual,46112,Petrol,20.0,60.1,1.4,audi
9,A1,2016,Manual,22451,Petrol,30.0,55.4,1.4,audi


Разделение на train и test

In [227]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8,  random_state = 42)

In [228]:
# проверяем на незаполненность данных и заполняем медианной
num_cols = ['year', 'mileage', 'tax', 'mpg', 'engineSize']
for col in num_cols:
    if col in X_train.columns:
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)


In [229]:
X_train

,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,brand
38628,Fiesta,2019,Manual,13282,Petrol,145.0,58.9,1.0,ford
42992,Kuga,2018,Semi-Auto,16276,Diesel,150.0,54.3,2.0,ford
46087,Mustang,2019,Automatic,12,Petrol,145.0,30.7,2.3,ford
12506,3 Series,2019,Semi-Auto,6250,Diesel,145.0,43.5,3.0,bmw
1118,A4,2016,Manual,29398,Petrol,145.0,51.4,1.4,audi
...,...,...,...,...,...,...,...,...,...
54886,GLA Class,2020,Semi-Auto,2500,Petrol,145.0,37.2,1.6,merc
76820,Aygo,2019,Manual,1094,Petrol,145.0,57.7,1.0,toyota
103694,Tiguan,2020,Manual,1700,Diesel,145.0,47.9,2.0,vw
860,A1,2020,Manual,556,Petrol,145.0,47.9,1.0,audi


Кодирование категориальных признаков

In [230]:
import category_encoders as ce
encoder = ce.one_hot.OneHotEncoder(cols=['model', 'transmission', 'fuelType', 'brand'])
X_train = encoder.fit_transform(X_train)
X_test = encoder.transform(X_test)

In [231]:
X_train.head(10)

,model_1,model_2,model_3,model_4,model_5,model_6,model_7,model_8,model_9,model_10,...,brand_2,brand_3,brand_4,brand_5,brand_6,brand_7,brand_8,brand_9,brand_10,brand_11
38628,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
42992,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
46087,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12506,0,0,0,1,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
1118,0,0,0,0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
681,0,0,0,0,0,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
78057,0,0,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
58957,0,0,0,0,0,0,0,1,0,0,...,0,0,0,1,0,0,0,0,0,0
21527,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
15968,0,0,0,0,0,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0


In [232]:
# Модель 1 - SGDRegressor
pipe_LR = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SGDRegressor(random_state=42, alpha=0.01, penalty='elasticnet', learning_rate='optimal'))
])


In [233]:
# Модель 2 - RandomForestRegressor
pipe_RF = Pipeline([
    ('model', RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42))
])


In [234]:
# Модель 3 - LGBMRegressor
from lightgbm import LGBMRegressor
pipe_LGBM = Pipeline([
    ('model', LGBMRegressor(n_estimators=100, max_depth=10, random_state=42, verbose=-1))
])
    

In [235]:
# Модель 4- StackingRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
stacking_model = StackingRegressor(
    estimators=[
        ('lr', pipe_LR),
        ('rf', pipe_RF),
        ('lgbm', pipe_LGBM)
    ],
    final_estimator=LinearRegression(),  
    cv=5,                                
    n_jobs=-1,
    passthrough=False                    
)

In [237]:
#Обучаем все модели и рассчитываем метрику r2_score
models = {
    'SGDRegressor': pipe_LR,
    'RandomForest': pipe_RF,
    'LightGBM': pipe_LGBM,
    'Stacking': stacking_model
}

results = []
for name, model in models.items():
    print(f" Обучение {name}...")
    model.fit(X_train, y_train)
    
    # Предсказания
    y_pred = model.predict(X_test)
    
    # Метрики
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'R²': r2        
    })
    print(f"  R²: {r2:.4f}")

 Обучение SGDRegressor...
  R²: 0.8499
 Обучение RandomForest...
  R²: 0.9515
 Обучение LightGBM...
  R²: 0.9416
 Обучение Stacking...
  R²: 0.9535


Кроссвалидация

In [238]:
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)
    cv_results[name] = scores
    print(f"{name:15s}: {scores.mean():.4f} (+/- {scores.std():.4f})")

SGDRegressor   : 0.8574 (+/- 0.0078)
RandomForest   : 0.9465 (+/- 0.0078)
LightGBM       : 0.9391 (+/- 0.0078)
Stacking       : 0.9497 (+/- 0.0075)


In [217]:
#Сравнение лучшей модели (RandomForest) и Stacking
ttest_rel(cv_results['Stacking'], cv_results['RandomForest'])

TtestResult(statistic=np.float64(11.174498819840956), pvalue=np.float64(0.0003650909404612828), df=np.int64(4))

pvalue = 0.0003650909404612828 < 0.001 - значит значимое различие между моделями Stacking и RandomForest